# TP3 — RAG multimodal agentique

Ce notebook suit les 7 étapes demandées avec le PDF **Attention Is All You Need**.
Les sorties générées sont placées dans `resultats_pdf/` (dossier ignoré par Git).

## Étape 1 — Extraction du texte et des images

In [1]:
from pathlib import Path
import hashlib
import json
import pymupdf

PDF_PATH = Path("pdfs") / "attention is all you need.pdf"
OUTPUT_DIR = Path("resultats_pdf")
TEXT_DIR = OUTPUT_DIR / "textes"
IMAGE_DIR = OUTPUT_DIR / "images"
OUTPUT_DIR.mkdir(exist_ok=True)
TEXT_DIR.mkdir(exist_ok=True)
IMAGE_DIR.mkdir(exist_ok=True)

assert PDF_PATH.exists(), f"PDF introuvable : {PDF_PATH}"
document_hash = hashlib.sha256(PDF_PATH.read_bytes()).hexdigest()

pdf = pymupdf.open(PDF_PATH)
pages = []
image_records = []
for page_number, page in enumerate(pdf, start=1):
    text = page.get_text("text").strip()
    text_path = TEXT_DIR / f"page_{page_number:03d}.txt"
    text_path.write_text(text, encoding="utf-8")
    pages.append({"page": page_number, "text": text, "path": str(text_path)})

    for image_number, image in enumerate(page.get_images(full=True), start=1):
        image_data = pdf.extract_image(image[0])
        image_path = IMAGE_DIR / f"page_{page_number:03d}_image_{image_number:02d}.{image_data['ext']}"
        image_path.write_bytes(image_data["image"])
        image_records.append({"page": page_number, "path": str(image_path)})

extraction_manifest = {
    "document": str(PDF_PATH),
    "sha256": document_hash,
    "pages": len(pages),
    "images": len(image_records),
}
(OUTPUT_DIR / "manifest.json").write_text(json.dumps(extraction_manifest, indent=2), encoding="utf-8")
print(f"Étape 1 terminée : {len(pages)} pages et {len(image_records)} images extraites.")

Étape 1 terminée : 11 pages et 3 images extraites.


Le texte est conservé page par page. Cette organisation garde la source (page et fichier image)
pour pouvoir expliquer les résultats du retriever.

## Étape 2 — Embeddings multimodaux avec CLIP

In [2]:
import numpy as np
import torch
import open_clip
from PIL import Image

MODEL_NAME = "ViT-B-32"
PRETRAINED = "laion2b_s34b_b79k"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

try:
    clip_model, _, preprocess = open_clip.create_model_and_transforms(
        MODEL_NAME, pretrained=PRETRAINED, device=DEVICE
    )
    tokenizer = open_clip.get_tokenizer(MODEL_NAME)
    clip_model.eval()
    EMBEDDING_BACKEND = "CLIP ViT-B-32"
except Exception as error:
    raise RuntimeError(
        "Le modèle CLIP n'a pas pu être chargé. Vérifiez la connexion pour le premier téléchargement."
    ) from error

@torch.no_grad()
def embed_text(texts):
    tokens = tokenizer(list(texts)).to(DEVICE)
    vectors = clip_model.encode_text(tokens)
    vectors = vectors / vectors.norm(dim=-1, keepdim=True).clamp_min(1e-12)
    return vectors.cpu().numpy().astype("float32")

@torch.no_grad()
def embed_image(paths):
    batch = torch.stack([preprocess(Image.open(path).convert("RGB")) for path in paths]).to(DEVICE)
    vectors = clip_model.encode_image(batch)
    vectors = vectors / vectors.norm(dim=-1, keepdim=True).clamp_min(1e-12)
    return vectors.cpu().numpy().astype("float32")

print(f"Étape 2 terminée avec {EMBEDDING_BACKEND} sur {DEVICE}.")

C:\Users\GiGi\Desktop\Activité Pratique N°3 - RAG Multi Modal Agentique\Activit-Pratique-N-3---RAG-Multi-Modal-Agentique\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Étape 2 terminée avec CLIP ViT-B-32 sur cpu.


## Étape 3 — Indexation dans ChromaDB

In [3]:
import chromadb

CHROMA_DIR = OUTPUT_DIR / "chroma"
client = chromadb.PersistentClient(path=str(CHROMA_DIR))
try:
    client.delete_collection("attention_multimodal")
except Exception:
    pass
collection = client.get_or_create_collection(
    name="attention_multimodal",
    metadata={"description": "Textes et images du PDF Attention Is All You Need"},
)

chunk_records = []
for page in pages:
    words = page["text"].split()
    chunk_size = 120
    overlap = 20
    for start in range(0, len(words), chunk_size - overlap):
        chunk = " ".join(words[start:start + chunk_size]).strip()
        if chunk:
            chunk_records.append({
                "id": f"{document_hash[:12]}-text-{page['page']}-{start}",
                "content": chunk,
                "metadata": {
                    "type": "text",
                    "page": page["page"],
                    "path": page["path"],
                    "document_sha256": document_hash,
                },
            })

text_vectors = embed_text([item["content"] for item in chunk_records])
collection.add(
    ids=[item["id"] for item in chunk_records],
    embeddings=text_vectors.tolist(),
    documents=[item["content"] for item in chunk_records],
    metadatas=[item["metadata"] for item in chunk_records],
)

if image_records:
    image_vectors = embed_image([item["path"] for item in image_records])
    collection.add(
        ids=[f"{document_hash[:12]}-image-{index}" for index, _ in enumerate(image_records)],
        embeddings=image_vectors.tolist(),
        documents=[f"Image extraite de la page {item['page']}" for item in image_records],
        metadatas=[{
            "type": "image",
            "page": item["page"],
            "path": item["path"],
            "document_sha256": document_hash,
        } for item in image_records],
    )

print(f"Étape 3 terminée : {collection.count()} éléments indexés.")

Étape 3 terminée : 59 éléments indexés.


## Étape 4 — Recherche multimodale

In [4]:
def search_multimodal(query, k=5):
    "Recherche par similarité cosinus pour une requête texte ou image."
    if isinstance(query, (str, Path)):
        query_path = Path(query)
        if query_path.exists():
            query_vector = embed_image([str(query_path)])[0]
        else:
            query_vector = embed_text([str(query)])[0]
    elif isinstance(query, Image.Image):
        temporary_image = OUTPUT_DIR / "_query_image.png"
        query.save(temporary_image)
        query_vector = embed_image([str(temporary_image)])[0]
    else:
        raise TypeError("La requête doit être un texte, un chemin d'image ou une image PIL.")

    result = collection.query(
        query_embeddings=[query_vector.tolist()],
        n_results=k,
        include=["documents", "metadatas", "distances"],
    )
    return [
        {"document": document, "metadata": metadata, "distance": distance}
        for document, metadata, distance in zip(
            result["documents"][0], result["metadatas"][0], result["distances"][0]
        )
    ]

text_results = search_multimodal("How does the Transformer use attention?", k=3)
for index, result in enumerate(text_results, start=1):
    print(f"{index}. page={result['metadata']['page']} type={result['metadata']['type']} "
          f"distance={result['distance']:.4f}")

1. page=8 type=text distance=0.5949
2. page=5 type=text distance=0.6508
3. page=2 type=text distance=0.7518


## Étape 5 — Retriever comme outil agentique

In [5]:
retriever_tool = {
    "name": "retriever_tool",
    "description": (
        "Recherche dans le PDF Attention Is All You Need. "
        "Utiliser cet outil pour obtenir les passages textuels et les images "
        "les plus proches d'une question."
    ),
    "input": "texte ou chemin vers une image",
}

def retriever_tool_run(query, k=5):
    return search_multimodal(query, k=k)

print(f"Outil disponible : {retriever_tool['name']}")
print(retriever_tool["description"])

Outil disponible : retriever_tool
Recherche dans le PDF Attention Is All You Need. Utiliser cet outil pour obtenir les passages textuels et les images les plus proches d'une question.


## Étape 6 — Agent ReAct avec LangGraph

In [6]:
import os
from dotenv import load_dotenv
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

load_dotenv()
llm = None
if os.getenv("GOOGLE_API_KEY"):
    from langchain_google_genai import ChatGoogleGenerativeAI
    llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0)

class AgentState(TypedDict, total=False):
    question: str
    action: str
    observation: list
    answer: str

def decide_action(state):
    return {"action": "retriever_tool"}

def retrieve_context(state):
    observation = retriever_tool_run(state["question"], k=4)
    return {"observation": observation}

def generate_answer(state):
    observations = state.get("observation", [])
    if not observations:
        answer = "Aucun contexte pertinent n'a été retrouvé dans le document."
    else:
        sources = ", ".join(
            f"page {item['metadata']['page']} ({item['metadata']['type']})"
            for item in observations
        )
        context = "\n".join(item["document"][:500] for item in observations)
        if llm is not None:
            prompt = (
                "Réponds en français à la question en utilisant uniquement le contexte suivant. "
                "Si l'information manque, dis-le clairement.\n\n"
                f"Question : {state['question']}\nContexte : {context}"
            )
            answer = str(llm.invoke(prompt).content)
        else:
            answer = f"Réponse fondée sur le document : {context[:440]}"
        answer += f" Sources : {sources}." 
    return {"answer": answer}

graph_builder = StateGraph(AgentState)
graph_builder.add_node("decide", decide_action)
graph_builder.add_node("retrieve", retrieve_context)
graph_builder.add_node("generate", generate_answer)
graph_builder.add_edge(START, "decide")
graph_builder.add_edge("decide", "retrieve")
graph_builder.add_edge("retrieve", "generate")
graph_builder.add_edge("generate", END)
agent = graph_builder.compile()

print("Graphe ReAct compilé : MIND (décision/réponse) -> BODY (retriever_tool).")

Graphe ReAct compilé : MIND (décision/réponse) -> BODY (retriever_tool).


## Étape 7 — Tests et validation

In [7]:
test_questions = [
    "Explain the role of attention in the Transformer.",
    "Which figures or diagrams in the paper are related to the Transformer architecture?",
]

validation_results = []
for question in test_questions:
    result = agent.invoke({"question": question})
    assert result["action"] == "retriever_tool"
    assert result["observation"], "Le retriever doit retourner au moins un résultat."
    assert result["answer"], "L'agent doit produire une réponse."
    validation_results.append(result)
    print(f"Question : {question}")
    print(result["answer"])
    print("-" * 80)

image_test = search_multimodal(image_records[0]["path"], k=2) if image_records else []
assert image_test, "La recherche image doit retourner un résultat."
print(f"Validation terminée : {len(validation_results)} tests texte et 1 test image.")

Question : Explain the role of attention in the Transformer.
Réponse fondée sur le document : our Model The Transformer uses multi-head attention in three different ways: • In "encoder-decoder attention" layers, the queries come from the previous decoder layer, and the memory keys and values come from the output of the encoder. This allows every position in the decoder to attend over all positions in the input sequence. This mimics the typical encoder-decoder attention mechanisms in sequence-to-sequence models such as [31, 2, 8] Sources : page 5 (text), page 8 (text), page 2 (text), page 2 (text).
--------------------------------------------------------------------------------
Question : Which figures or diagrams in the paper are related to the Transformer architecture?
Réponse fondée sur le document : ConvS2S Ensemble [8] 26.36 41.29 7.7 · 1019 1.2 · 1021 Transformer (base model) 27.3 38.1 3.3 · 1018 Transformer (big) 28.4 41.0 2.3 · 1019 Label Smoothing During training, we employed l

### Conclusion

Le pipeline est complet : extraction PDF, embeddings CLIP normalisés,
indexation ChromaDB avec métadonnées, recherche texte/image, outil agentique,
graphe LangGraph et tests de validation.

Gemini est utilisé si `GOOGLE_API_KEY` est défini dans le fichier `.env`.
Sans clé, le notebook utilise automatiquement une réponse locale de secours.
La clé n'est jamais écrite dans le notebook ni envoyée dans Git.
